# Merge and Window Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

patient_silver = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/silver/patients"
)

visit_silver = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/silver/visits"
)


## Apply Lag and Lead

In [0]:
visit_silver.printSchema()

root
 |-- visit_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: double (nullable = true)
 |-- load_ts: timestamp (nullable = true)



In [0]:
doctor_revenue = (
    visit_silver
    .groupBy("doctor_id")
    .agg(
        sum("bill_amount")
        .alias("revenue_generated")
    )
)

In [0]:
revenue_trend = (
    visit_silver
    .groupBy("visit_date")
    .agg(
        sum("bill_amount").alias("daily_revenue")
    )
)

date_window = Window.orderBy("visit_date")

revenue_trend = (
    revenue_trend
    .withColumn(
        "previous_day_revenue",
        lag("daily_revenue").over(date_window)
    )
    .withColumn(
        "next_day_revenue",
        lead("daily_revenue").over(date_window)
    )
)

display(revenue_trend)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


visit_date,daily_revenue,previous_day_revenue,next_day_revenue
2025-01-01,1301210.2699999996,null,1264129.9999999998
2025-01-02,1264129.9999999998,1301210.2699999996,1304322.1200000003
2025-01-03,1304322.1200000003,1264129.9999999998,1228262.37
2025-01-04,1228262.37,1304322.1200000003,1213125.5600000015
2025-01-05,1213125.5600000015,1228262.37,1264203.6600000004
2025-01-06,1264203.6600000004,1213125.5600000015,1138736.0199999996
2025-01-07,1138736.0199999996,1264203.6600000004,1139231.2799999993
2025-01-08,1139231.2799999993,1138736.0199999996,1233442.2300000011
2025-01-09,1233442.2300000011,1139231.2799999993,1271413.560000001
2025-01-10,1271413.560000001,1233442.2300000011,1270969.9199999983


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# display(revenue_trend)

## Delta Merge Demo

In [0]:
new_doctor_data = spark.createDataFrame(
    [
        ("DOC0001",80000),
        ("DOC0002",90000)
    ],
    ["doctor_id","revenue_generated"]
)

new_doctor_data.createOrReplaceTempView(
    "new_doctor_data"
)

In [0]:
doctor_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/merge_demo")

In [0]:
spark.read.format("delta") \
.load("/Volumes/workspace/default/final_assignment/merge_demo") \
.createOrReplaceTempView("doctor_revenue_target")

## Merge

In [0]:
%sql
MERGE INTO delta.`/Volumes/workspace/default/final_assignment/merge_demo` AS target
using new_doctor_data as source
on target.doctor_id = source.doctor_id

when matched then 
update set 
target.revenue_generated = source.revenue_generated

when not matched then insert(
    doctor_id, revenue_generated
)
values(
    source.doctor_id, source.revenue_generated
)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/merge_demo")
)

doctor_id,revenue_generated
PRO00798,44303.37
PRO00918,189532.03999999998
PRO00371,53387.54000000001
PRO00028,316876.91
PRO00045,295421.50999999995
PRO01126,60371.030000000006
PRO00889,198515.13
PRO00007,372621.1400000001
PRO01116,75365.26000000001
PRO01088,35958.63


In [0]:
# dbutils.fs.rm(
#     "/Volumes/workspace/default/final_assignment/gold/doctor_revenue_merge",
#     True
# )